# Unit 3: Deep Q-Learning with Atari Games 👾 using RL Baselines3 Zoo

<div align="center">
  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/thumbnail.jpg" alt="Deep Q-Learning Space Invaders" width="100%"/>
</div>

<br>

<div align="center">
    <a href="https://github.com/DLR-RM/rl-baselines3-zoo"><img src="https://img.shields.io/badge/Framework-RL%20Baselines3%20Zoo-blue?style=for-the-badge" alt="RL Zoo"></a>
    <a href="https://stable-baselines3.readthedocs.io/"><img src="https://img.shields.io/badge/Library-Stable--Baselines3-brightgreen?style=for-the-badge" alt="SB3"></a>
    <a href="https://gymnasium.farama.org/"><img src="https://img.shields.io/badge/Environment-Gymnasium%20(Atari)-orange?style=for-the-badge" alt="Gym"></a>
</div>

<br>

---

## Project Overview

In this project, I trained a **Deep Q-Network (DQN)** agent to master the classic Atari game **[Space Invaders](https://ale.farama.org/environments/space_invaders/)**.

This work is based on **[Unit 3 of the Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit3/introduction)**. Unlike traditional Q-Learning (which uses tables), this agent uses a **Convolutional Neural Network (CNN)** to process raw pixel data from the game screen, allowing it to "see" and react to the environment in real-time. The training was conducted using the **RL Baselines3 Zoo** framework, a robust pipeline for reproducible Reinforcement Learning experiments.

<br>

### 🎯 Key Objectives

* **Implementation:** Deploy a DQN agent capable of handling high-dimensional observation spaces (images).
* **Environment:** Solve the `SpaceInvadersNoFrameskip-v4` environment from the Arcade Learning Environment (ALE).
* **Performance:** Tune hyperparameters to achieve a competitive score and record gameplay footage.
* **Deployment:** Push the trained model and metrics to the Hugging Face Hub for public evaluation.

<br>

### 🛠️ Tech Stack & Methodology

* **Algorithm:** Deep Q-Learning (DQN) with Convolutional Layers.
* **Frameworks:** `Stable-Baselines3`, `RL Baselines3 Zoo`, `Gymnasium`.
* **Input:** Raw pixel frames (preprocessed and stacked).
* **Output:** Discrete joystick actions.

<br>

---

## Install RL-Baselines3 Zoo and its dependencies 📚

In [2]:
!pip install git+https://github.com/DLR-RM/rl-baselines3-zoo

  Cloning https://github.com/DLR-RM/rl-baselines3-zoo to /tmp/pip-req-build-h3du4zuy
  Running command git clone --filter=blob:none --quiet https://github.com/DLR-RM/rl-baselines3-zoo /tmp/pip-req-build-h3du4zuy
  Resolved https://github.com/DLR-RM/rl-baselines3-zoo to commit d29756c456caadbbebc15c35893674abb2453e0d
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.3/187.3 kB 21.2 MB/s eta 0:00:00
  Created wheel for rl_zoo3: filename=rl_zoo3-2.8.0a0-py3-none-any.whl size=78003 sha256=5db9b121077dbb271c9c3d748360e68cbb8d1772feac7d6e6009d9594be04f26
  Stored in 

In [3]:
!apt-get install swig cmake ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


To be able to use Atari games in Gymnasium we need to install atari package. And accept-rom-license to download the rom files (games files).

In [4]:
!pip install gymnasium[atari]
!pip install gymnasium[accept-rom-license]

## Create a virtual display 🔽

During the notebook, we'll need to generate a replay video. To do so, with colab, **we need to have a virtual screen to be able to render the environment** (and thus record the frames).

Hence the following cell will install the librairies and create and run a virtual screen 🖥

In [5]:
%%capture
!apt install python-opengl
!apt install xvfb
!pip3 install pyvirtualdisplay

In [6]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## Train our Deep Q-Learning Agent to Play Space Invaders 👾

To train an agent with RL-Baselines3-Zoo, we just need to do two things:

1. Create a hyperparameter config file that will contain our training hyperparameters called `dqn.yml`.

This is a template example:

```
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: 'CnnPolicy'
  n_timesteps: !!float 1e6
  buffer_size: 100000
  learning_rate: !!float 1e-4
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  # If True, you need to deactivate handle_timeout_termination
  # in the replay_buffer_kwargs
  optimize_memory_usage: False
```

Here we see that:
- We use the `Atari Wrapper` that preprocess the input (Frame reduction ,grayscale, stack 4 frames)
- We use `CnnPolicy`, since we use Convolutional layers to process the frames
- We train it for 1 million `n_timesteps`
- Memory (Experience Replay) size is 100000, aka the amount of experience steps you saved to train again your agent with.

In [7]:
!nvidia-smi

Mon Jan 26 16:02:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In terms of hyperparameters optimization, my advice is to focus on these 3 hyperparameters:
- `learning_rate`
- `buffer_size (Experience Memory size)`
- `batch_size`

As a good practice, you need to **check the documentation to understand what each hyperparameters does**: https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#parameters



2. We start the training and save the models on `logs` folder 📁

- Define the algorithm after `--algo`, where we save the model after `-f` and where the hyperparameter config is after `-c`.

In [9]:
!python -m rl_zoo3.train --algo dqn  --env SpaceInvadersNoFrameskip-v4 -f logs/ -c dqn.yml

Streaming output truncated to the last 5000 lines.
| rollout/            |          |
|    ep_len_mean      | 2.9e+03  |
|    ep_rew_mean      | 351      |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 2676     |
|    fps              | 275      |
|    time_elapsed     | 2023     |
|    total_timesteps  | 557163   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0232   |
|    n_updates        | 114290   |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.9e+03  |
|    ep_rew_mean      | 350      |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 2680     |
|    fps              | 275      |
|    time_elapsed     | 2025     |
|    total_timesteps  | 557768   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0122   |
|   

## Let's evaluate our agent 👀
- RL-Baselines3-Zoo provides `enjoy.py`, a python script to evaluate our agent. In most RL libraries, we call the evaluation script `enjoy.py`.
- Let's evaluate it for 5000 timesteps 🔥

In [10]:
!python -m rl_zoo3.enjoy  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --no-render  --n-timesteps 5000  --folder logs/

2026-01-26 17:16:52.327774: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769447812.429876   30399 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769447812.447966   30399 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769447812.511148   30399 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769447812.511190   30399 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769447812.511200   30399 computation_placer.cc:177] computation placer alr

## Publish our trained model on the Hub 🚀
Now that we saw we got good results after the training, we can publish our trained model on the hub 🤗 with one line of code.

In [11]:
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.
notebook_login()
!git config --global credential.helper store

In [12]:
!python -m rl_zoo3.push_to_hub  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --repo-name dqn-SpaceInvadersNoFrameskip-v4  -orga MykhailoMatsyshyn  -f logs/

2026-01-26 17:21:52.075355: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769448112.096426   31706 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769448112.106214   31706 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769448112.128418   31706 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769448112.128451   31706 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769448112.128457   31706 computation_placer.cc:177] computation placer alr

In [13]:
!python -m rl_zoo3.record_video --algo dqn --env SpaceInvadersNoFrameskip-v4 -f logs/ -n 2000

2026-01-26 17:35:08.322628: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769448908.342689   35082 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769448908.348817   35082 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769448908.364403   35082 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769448908.364429   35082 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769448908.364433   35082 computation_placer.cc:177] computation placer alr

In [14]:
!ls -R logs/dqn/SpaceInvadersNoFrameskip-v4_1/videos/

logs/dqn/SpaceInvadersNoFrameskip-v4_1/videos/:
final-model-dqn-SpaceInvadersNoFrameskip-v4-step-0-to-step-2000.mp4


In [15]:
!huggingface-cli upload MykhailoMatsyshyn/dqn-SpaceInvadersNoFrameskip-v4 \
logs/dqn/SpaceInvadersNoFrameskip-v4_1/videos/final-model-dqn-SpaceInvadersNoFrameskip-v4-step-0-to-step-2000.mp4 \
replay.mp4 \
--repo-type model

⚠️  Warning: 'huggingface-cli upload' is deprecated. Use 'hf upload' instead.
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...4-step-0-to-step-2000.mp4: 100% 421k/421k [00:00<?, ?B/s]

Processing Files (1 / 1)      : 100% 421k/421k [00:00<00:00, 519kB/s,  702kB/s  ]
New Data Upload               : 100% 421k/421k [00:00<00:00, 519kB/s,  702kB/s  ]

  ...4-step-0-to-step-2000.mp4: 100% 421k/421k [00:00<?, ?B/s]

Processing Files (1 / 1)      : 100% 421k/421k [00:01<00:00, 416kB/s,  527kB/s  ]
New Data Upload               : 100% 421k/421k [00:01<00:00, 417kB/s,  527kB/s  ]
  ...4-step-0-to-step-2000.mp4: 100% 421k/421k [00:00<?, ?B/s]
https://huggingface.co/MykhailoMatsyshyn/dqn-SpaceInvadersNoFrameskip-v4/blob/main/replay.mp4


## Load a powerful trained model 🔥
- The Stable-Baselines3 team uploaded **more than 150 trained Deep Reinforcement Learning agents on the Hub**.

You can find them here: 👉 https://huggingface.co/sb3

Some examples:
- Asteroids: https://huggingface.co/sb3/dqn-AsteroidsNoFrameskip-v4
- Beam Rider: https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4
- Breakout: https://huggingface.co/sb3/dqn-BreakoutNoFrameskip-v4
- Road Runner: https://huggingface.co/sb3/dqn-RoadRunnerNoFrameskip-v4

Let's load an agent playing Beam Rider: https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4

1. We download the model using `rl_zoo3.load_from_hub`, and place it in a new folder that we can call `rl_trained`

In [16]:
# Download model and save it into the logs/ folder
!python -m rl_zoo3.load_from_hub --algo dqn --env BeamRiderNoFrameskip-v4 -orga sb3 -f rl_trained/

2026-01-26 17:40:40.573414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769449240.594066   36521 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769449240.600501   36521 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769449240.616498   36521 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769449240.616532   36521 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769449240.616536   36521 computation_placer.cc:177] computation placer alr

2. Let's evaluate if for 5000 timesteps

In [17]:
!python -m rl_zoo3.enjoy --algo dqn --env BeamRiderNoFrameskip-v4 -n 5000  -f rl_trained/ --no-render

2026-01-26 17:41:13.489910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769449273.510597   36675 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769449273.516659   36675 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769449273.531844   36675 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769449273.531872   36675 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769449273.531876   36675 computation_placer.cc:177] computation placer alr

In [19]:
!python -m rl_zoo3.load_from_hub --algo dqn --env BeamRiderNoFrameskip-v4 -orga sb3 -f rl_trained/

!python -m rl_zoo3.record_video --algo dqn --env BeamRiderNoFrameskip-v4 -f rl_trained/ -n 2000 -o video_folder/

2026-01-26 17:51:30.624262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769449890.646048   39469 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769449890.652240   39469 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769449890.667674   39469 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769449890.667702   39469 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769449890.667706   39469 computation_placer.cc:177] computation placer alr

## Future Work: From Space Invaders to BeamRider

Now that I've successfully trained a Deep Q-Network agent to master **Space Invaders** (achieving a mean reward of ~490), my next objective is to tackle a more complex environment: **[BeamRiderNoFrameskip-v4](https://gymnasium.farama.org/environments/atari/beam_rider/)**.

<br>

### The Challenge: Hyperparameter Tuning
While training this agent, I utilized a standard set of hyperparameters. However, I recognize that manual tuning (trial and error) is inefficient and often suboptimal. Sensitivity to hyperparameters like `learning_rate` and `buffer_size` is a well-known challenge in Reinforcement Learning.

<br>

### Next Steps
To improve upon this baseline and build a more robust pipeline, my next project will focus on **Automated Hyperparameter Optimization**.

I plan to integrate **[Optuna](https://optuna.org/)**, a hyperparameter optimization framework, to systematically search for the best configuration. This will allow me to:
1.  Automate the search for the "sweet spot" in parameters.
2.  Maximize the agent's performance with efficient pruning of unpromising trials.
3.  Apply this optimized pipeline to solve BeamRider effectively.